<a href="https://colab.research.google.com/github/M-coder-web-cell/llm-engineering/blob/Mrinmoy/PythonCodeToCppPort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Dependencies

First, we'll install the required libraries for this notebook.

In [ ]:
# Install python-dotenv to load environment variables from a .env file
!pip install python-dotenv openai

In [ ]:
from openai import OpenAI

In [ ]:
#for openai compatible endpoint OpenAI(api_key, base_url)

gemini_base_url = "https://api.anthropic.com/v1/"
gemini_api_key = "YOUR_API_KEY"
client = OpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [ ]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
Respond only with C++ code.
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages(python):
  return [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt_for(python)}
  ]

In [ ]:
def generate_code(python_code):
  response = client.chat.completions.create(model = "claude-sonnet-4-6", messages = messages(python_code))
  content = response.choices[0].message.content
  content = content.replace("```cpp", "").replace("```", "")
  with open("main.cpp", "w") as file:
    file.write(content)

In [ ]:
python = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [ ]:
import sys
import io

def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
import subprocess

def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
compile_and_run()

In [ ]:
generate_code(python_code=python)